# 1 — Préparation des données

Nettoyage, alignement des **deux cohortes** (pédiatrique + clinique), feature
engineering clinique (indice de Mentzer), contrôle qualité, split et
standardisation.

**Produit :**
- `data/processed/clinical_clean.csv`, `pediatric_clean.csv`
- `data/processed/datasets.joblib` (tableaux prêts pour l'entraînement)
- `models/scaler.joblib`

> Prérequis : les fichiers Excel bruts dans `data/` (voir `run_all.ps1`).

In [ ]:
import os, sys
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Rendre le module de configuration importable.
SRC = Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd() / "src"
sys.path.append(str(SRC))
import config as C

np.random.seed(C.RANDOM_SEED)
print("Features modèle :", C.MODEL_FEATURES)

## Fonctions de nettoyage

Normalisation du sexe, coercition numérique + imputation médiane, garde-fou
physiologique, et journalisation des problèmes qualité rencontrés.

In [ ]:
def _normalize_sex(series):
    s = series.astype(str).str.strip().str.upper().str[0]
    return s.where(s.isin(["F", "M"]), other=pd.NA)


def _find_pretrain_file():
    files = sorted(C.PRETRAIN_DIR.rglob("*.xlsx"))
    if not files:
        raise FileNotFoundError(f"Aucun .xlsx sous {C.PRETRAIN_DIR}")
    return files[0]


def load_and_clean(path, rename_map, source):
    """Charge un Excel, aligne le schéma, nettoie et journalise la qualité."""
    issues = []
    df = pd.read_excel(path)
    n0 = len(df)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns={k.strip(): v for k, v in rename_map.items()})

    missing = [c for c in ([C.TARGET] + C.BASE_FEATURES) if c not in df.columns]
    if missing:
        raise ValueError(f"[{source}] colonnes manquantes : {missing}")
    df = df[C.BASE_FEATURES + [C.TARGET]].copy()

    # Sexe
    df["sex_label"] = _normalize_sex(df["sex"])
    n_bad = int(df["sex_label"].isna().sum())
    if n_bad:
        issues.append({"source": source, "probleme": "sexe non reconnu", "nb": n_bad,
                       "action": "imputation mode"})
        df["sex_label"] = df["sex_label"].fillna(df["sex_label"].mode(dropna=True)[0])
    df["sex"] = (df["sex_label"] == "M").astype(int)

    # Numériques : coercition + imputation médiane
    for col in ["age", "hgb", "rbc", "hct", "mcv", "mch", "mchc"]:
        coerced = pd.to_numeric(df[col], errors="coerce")
        nb = int(coerced.isna().sum())
        if nb:
            issues.append({"source": source, "probleme": f"'{col}' non numérique/manquant",
                           "nb": nb, "action": "imputation médiane"})
            coerced = coerced.fillna(coerced.median())
        df[col] = coerced

    # Garde-fou physiologique
    for col, (lo, hi) in C.PLAUSIBLE_RANGES.items():
        mask = (df[col] < lo) | (df[col] > hi)
        nb = int(mask.sum())
        if nb:
            issues.append({"source": source, "probleme": f"'{col}' hors plage [{lo},{hi}]",
                           "nb": nb, "action": "remplacé par médiane"})
            df.loc[mask, col] = df[col].median()

    # Cible + feature dérivée (indice de Mentzer)
    df[C.TARGET] = pd.to_numeric(df[C.TARGET], errors="coerce").fillna(0).astype(int).clip(0, 1)
    df["mentzer_index"] = df["mcv"] / df["rbc"].replace(0, np.nan)
    df["mentzer_index"] = df["mentzer_index"].fillna(df["mentzer_index"].median())
    df["source"] = source
    print(f"[{source}] {n0} -> {len(df)} lignes | positifs {df[C.TARGET].mean():.1%}")
    return df, issues

## Charger et nettoyer les deux cohortes

In [ ]:
clin, iss_c = load_and_clean(C.CLINICAL_XLSX, C.CLINICAL_RENAME, "clinical")
pedia, iss_p = load_and_clean(_find_pretrain_file(), C.PRETRAIN_RENAME, "pediatric")

# Contrôle de fuite : chevauchement exact entre cohortes.
overlap = clin[C.BASE_FEATURES].merge(
    pedia[C.BASE_FEATURES].drop_duplicates(), on=C.BASE_FEATURES, how="inner")
n_overlap = len(overlap)
print("Chevauchement inter-cohortes :", n_overlap, "(0 = pas de fuite)")

## Splits stratifiés + standardisation

Le `StandardScaler` est ajusté sur **l'union des trains** (pédiatrique + clinique)
— jamais sur le test — pour éviter toute fuite.

In [ ]:
clin_tv, clin_test = train_test_split(clin, test_size=C.TEST_SIZE,
                                      stratify=clin[C.TARGET], random_state=C.RANDOM_SEED)
clin_train, clin_val = train_test_split(clin_tv, test_size=C.VAL_SIZE,
                                        stratify=clin_tv[C.TARGET], random_state=C.RANDOM_SEED)
pedia_train, pedia_val = train_test_split(pedia, test_size=C.VAL_SIZE,
                                          stratify=pedia[C.TARGET], random_state=C.RANDOM_SEED)

scaler = StandardScaler().fit(
    pd.concat([pedia_train[C.CONTINUOUS_FEATURES], clin_train[C.CONTINUOUS_FEATURES]]))


def to_X(frame):
    out = frame[C.MODEL_FEATURES].copy()
    out[C.CONTINUOUS_FEATURES] = scaler.transform(out[C.CONTINUOUS_FEATURES])
    return out[C.MODEL_FEATURES].to_numpy(dtype="float32")


data = {
    "feature_names": C.MODEL_FEATURES,
    "X_pre_train": to_X(pedia_train), "y_pre_train": pedia_train[C.TARGET].to_numpy("float32"),
    "X_pre_val": to_X(pedia_val), "y_pre_val": pedia_val[C.TARGET].to_numpy("float32"),
    "X_clin_train": to_X(clin_train), "y_clin_train": clin_train[C.TARGET].to_numpy("float32"),
    "X_clin_val": to_X(clin_val), "y_clin_val": clin_val[C.TARGET].to_numpy("float32"),
    "X_clin_test": to_X(clin_test), "y_clin_test": clin_test[C.TARGET].to_numpy("float32"),
    "clin_test_df": clin_test.reset_index(drop=True), "n_overlap": int(n_overlap),
}
print("Splits -> pre_train=%d clin_train=%d clin_val=%d clin_test=%d" % (
    len(data["y_pre_train"]), len(data["y_clin_train"]),
    len(data["y_clin_val"]), len(data["y_clin_test"])))

## Sauvegarde des artefacts

On tague chaque patient clinique par son split (utile pour filtrer sur le test)
puis on sauvegarde les CSV, le scaler et le `datasets.joblib`.

In [ ]:
clin = clin.copy()
clin["split"] = "train"
clin.loc[clin_val.index, "split"] = "val"
clin.loc[clin_test.index, "split"] = "test"

clin.to_csv(C.PROCESSED_DIR / "clinical_clean.csv", index=False)
pedia.to_csv(C.PROCESSED_DIR / "pediatric_clean.csv", index=False)
joblib.dump(scaler, C.MODELS_DIR / "scaler.joblib")
joblib.dump(data, C.PROCESSED_DIR / "datasets.joblib")

issues = iss_c + iss_p
print(f"OK — {len(clin)} patients cliniques, {len(pedia)} pédiatriques.")
print(f"Problèmes qualité journalisés : {len(issues)}")
print("Artefacts ->", C.PROCESSED_DIR)